# 03 — Customer Segmentation & Cohort Retention Analysis

This notebook answers two questions that sit at the core of any retention strategy:
who are our customers right now (RFM segmentation), and are we keeping them
(cohort retention). The outputs feed directly into the CLV model in notebook 04
and inform which segments should receive win-back campaigns.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..').resolve()))

from src.data_processing import load_processed
from src.segmentation import (
    compute_rfm,
    segment_summary,
    plot_rfm_treemap,
    plot_rfm_scatter,
    plot_monetary_distribution,
)

PROC_PATH = Path('../data/processed/orders_master.parquet')
FIG_DIR   = Path('../outputs/figures')
TBL_DIR   = Path('../outputs/tables')

pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.max_columns', 40)

df = load_processed(PROC_PATH)
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Unique customers: {df["customer_unique_id"].nunique():,}')
df.head(3)

## 1. RFM Segmentation

RFM stands for Recency, Frequency, and Monetary value. Each customer gets a
score from 1 to 5 on each dimension based on quintiles, and the three scores
are combined into a named segment like Champions, At Risk, or Lost. This gives
us a compact, explainable picture of the customer base without any machine
learning — every segment maps directly to a marketing action.

In [ ]:
rfm = compute_rfm(df)
print(f'RFM table: {rfm.shape[0]:,} customers')
print(f'Segments : {rfm["segment"].nunique()} unique')
rfm.head()

In [ ]:
summary = segment_summary(rfm)
summary.to_csv(TBL_DIR / '03_segment_summary.csv', index=False)
print(f'Saved → {TBL_DIR / "03_segment_summary.csv"}')
summary

In [ ]:
plot_rfm_treemap(rfm, path=FIG_DIR / '03_rfm_treemap.png')

In [ ]:
plot_rfm_scatter(rfm, path=FIG_DIR / '03_rfm_scatter.png')

In [ ]:
plot_monetary_distribution(rfm, path=FIG_DIR / '03_rfm_monetary_dist.png')

In [ ]:
# RFM score distributions per segment — shows how tight the scoring is
from src.segmentation import SEGMENT_COLORS

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
score_cols = [('r_score', 'Recency Score'), ('f_score', 'Frequency Score'), ('m_score', 'Monetary Score')]

seg_order = summary['segment'].tolist()

for ax, (col, title) in zip(axes, score_cols):
    data   = [rfm.loc[rfm['segment'] == seg, col].values for seg in seg_order]
    colors = [SEGMENT_COLORS.get(seg, '#bdc3c7') for seg in seg_order]
    bp = ax.boxplot(data, patch_artist=True, showfliers=False,
                    medianprops=dict(color='black', linewidth=1.5))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    ax.set_xticks(range(1, len(seg_order) + 1))
    ax.set_xticklabels(seg_order, rotation=35, ha='right', fontsize=8)
    ax.set_ylabel('Score (1–5)', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylim(0.5, 5.5)
    ax.grid(True, axis='y', linestyle='--', alpha=0.4)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('RFM Score Distributions by Segment', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / '03_rfm_score_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Cohort Retention Heatmap

A cohort here means all customers who placed their first order in the same
calendar month. We track what fraction of each cohort came back to buy again
one month later, two months later, and so on. The result is a heatmap where
each row is an acquisition cohort, each column is months since first purchase,
and the colour shows the retention rate. Darker cells mean more customers returned.
The diagonal fade from left to right is normal — retention drops over time.
What we look for is whether any cohort rows stand out as unusually high or low.

In [ ]:
# df is already order-level (one row per order) with cohort_month and cohort_index
cohort_df = df[['customer_unique_id', 'cohort_month', 'cohort_index']].dropna()

# Cohort sizes: unique customers in their first month (index == 0)
cohort_sizes = (
    cohort_df[cohort_df['cohort_index'] == 0]
    .groupby('cohort_month')['customer_unique_id']
    .nunique()
    .rename('cohort_size')
)

# Active customers per cohort per month offset
cohort_counts = (
    cohort_df
    .groupby(['cohort_month', 'cohort_index'])['customer_unique_id']
    .nunique()
    .rename('active')
    .reset_index()
)

cohort_counts = cohort_counts.join(cohort_sizes, on='cohort_month')
cohort_counts['retention'] = cohort_counts['active'] / cohort_counts['cohort_size']

retention_pivot = (
    cohort_counts
    .pivot_table(index='cohort_month', columns='cohort_index', values='retention')
    .sort_index()
)

# Trim to first 13 cohort months and drop cohorts with no index-0 data
retention_pivot = retention_pivot.loc[
    retention_pivot.index.isin(cohort_sizes[cohort_sizes >= 30].index),
    [c for c in range(13) if c in retention_pivot.columns],
]

print(f'Cohort matrix: {retention_pivot.shape[0]} cohorts x {retention_pivot.shape[1]} time steps')
retention_pivot.round(3).head(6)

In [ ]:
retention_pivot.to_csv(TBL_DIR / '03_cohort_retention_matrix.csv')
print(f'Saved → {TBL_DIR / "03_cohort_retention_matrix.csv"}')

annot_df = retention_pivot.applymap(lambda v: f'{v*100:.1f}%' if pd.notna(v) else '')

vmax = float(retention_pivot.drop(columns=0, errors='ignore').max().max())
vmax = min(round(vmax + 0.02, 2), 1.0)

fig, ax = plt.subplots(figsize=(15, max(6, len(retention_pivot) * 0.42)))
sns.heatmap(
    retention_pivot,
    annot=annot_df,
    fmt='',
    cmap='YlOrRd_r',
    linewidths=0.3,
    linecolor='#f0f0f0',
    ax=ax,
    vmin=0,
    vmax=vmax,
    cbar_kws={'label': 'Retention Rate', 'shrink': 0.6},
    annot_kws={'size': 7.5},
)
ax.set_title(
    'Monthly Cohort Retention Heatmap\n'
    'Row = acquisition cohort   |   Column = months since first purchase',
    fontsize=13, fontweight='bold', pad=12,
)
ax.set_xlabel('Months Since First Purchase', fontsize=11)
ax.set_ylabel('Acquisition Cohort', fontsize=11)
ax.set_xticklabels(
    [f'M{int(c)}' for c in retention_pivot.columns],
    rotation=0, fontsize=9,
)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / '03_cohort_retention_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Average Retention Curve

The heatmap shows individual cohorts but makes it hard to see the overall shape
of retention. By averaging across all cohorts at each time step we get a single
curve that shows the typical drop-off pattern. This is the number a growth team
would put in a weekly business review: how many customers are still active one,
three, and six months after their first purchase.

In [ ]:
avg_retention = retention_pivot.mean() * 100
std_retention = retention_pivot.std() * 100

ret_summary = pd.DataFrame({
    'cohort_index':    avg_retention.index,
    'avg_retention_pct': avg_retention.values.round(2),
    'std_pct':         std_retention.values.round(2),
    'n_cohorts':       retention_pivot.notna().sum().values,
})
ret_summary.to_csv(TBL_DIR / '03_avg_retention_curve.csv', index=False)
print(f'Saved → {TBL_DIR / "03_avg_retention_curve.csv"}')
ret_summary

In [ ]:
x   = avg_retention.index.tolist()
y   = avg_retention.values
err = std_retention.values

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(x, y, color='#e74c3c', linewidth=2.5, marker='o', markersize=6, zorder=3)
ax.fill_between(x, y - err, y + err, alpha=0.15, color='#e74c3c', label='±1 std dev')
ax.fill_between(x, 0, y, alpha=0.08, color='#e74c3c')

# Annotate key milestones
for month in [1, 3, 6]:
    if month in x:
        val = avg_retention[month]
        ax.annotate(
            f'M{month}: {val:.1f}%',
            xy=(month, val),
            xytext=(month + 0.2, val + 0.5),
            fontsize=9, color='#c0392b',
            arrowprops=dict(arrowstyle='->', color='#c0392b', lw=1),
        )

ax.set_xlabel('Months Since First Purchase', fontsize=11)
ax.set_ylabel('Avg Retention Rate (%)', fontsize=11)
ax.set_title('Average Cohort Retention Curve  (shaded band = ±1 std dev across cohorts)',
             fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.1f}%'))
ax.set_xticks(x)
ax.legend(frameon=False, fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / '03_avg_retention_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Individual cohort curves overlaid — highlights best and worst cohorts
fig, ax = plt.subplots(figsize=(12, 5))

palette   = plt.cm.Blues(np.linspace(0.3, 0.9, len(retention_pivot)))
cohorts   = retention_pivot.index.tolist()

for i, cohort in enumerate(cohorts):
    row = retention_pivot.loc[cohort].dropna() * 100
    ax.plot(row.index, row.values, color=palette[i], linewidth=1, alpha=0.6)

ax.plot(x, y, color='#e74c3c', linewidth=3, label='Average', zorder=5)

sm = plt.cm.ScalarMappable(cmap='Blues',
                            norm=plt.Normalize(vmin=0, vmax=len(cohorts)))
sm.set_array([])
cb = plt.colorbar(sm, ax=ax, label='Cohort (older → newer)', shrink=0.7)
cb.set_ticks([])

ax.set_xlabel('Months Since First Purchase', fontsize=11)
ax.set_ylabel('Retention Rate (%)', fontsize=11)
ax.set_title('Retention Curves — All Cohorts  (red = average)', fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:.1f}%'))
ax.set_xticks(x)
ax.legend(frameon=False, fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / '03_all_cohort_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Repeat Purchase Rate by First Purchase Category

Not all product categories are equal as customer acquisition channels. A customer
who first buys a phone accessory may be far more likely to return than one who
first buys a piece of furniture. This section identifies which first-purchase
categories produce the stickiest customers — useful for deciding where to run
acquisition campaigns and which categories deserve prominent placement for
new visitors.

In [ ]:
# First purchase category per customer
first_cat = (
    df.sort_values('order_purchase_timestamp')
    .groupby('customer_unique_id')['product_category']
    .first()
    .reset_index(name='first_category')
)

# Total orders per customer
order_counts = (
    df.groupby('customer_unique_id')['order_id']
    .count()
    .reset_index(name='order_count')
)

# Total spend per customer
spend = (
    df.groupby('customer_unique_id')['revenue']
    .sum()
    .reset_index(name='total_spend')
)

cust_cat = first_cat.merge(order_counts, on='customer_unique_id').merge(spend, on='customer_unique_id')
cust_cat['is_repeat'] = (cust_cat['order_count'] > 1).astype(int)

cat_retention = (
    cust_cat.dropna(subset=['first_category'])
    .groupby('first_category')
    .agg(
        customer_count  = ('customer_unique_id', 'count'),
        repeat_rate     = ('is_repeat', 'mean'),
        avg_orders      = ('order_count', 'mean'),
        avg_total_spend = ('total_spend', 'mean'),
    )
    .reset_index()
    .query('customer_count >= 50')
    .sort_values('repeat_rate', ascending=False)
    .reset_index(drop=True)
)

cat_retention['repeat_rate_pct'] = (cat_retention['repeat_rate'] * 100).round(2)
cat_retention.to_csv(TBL_DIR / '03_category_repeat_rate.csv', index=False)
print(f'Saved → {TBL_DIR / "03_category_repeat_rate.csv"}')
cat_retention.head(10)

In [ ]:
top_n  = cat_retention.head(20)
bot_n  = cat_retention.tail(10).sort_values('repeat_rate')
global_rate = cust_cat['is_repeat'].mean() * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Top 20 — highest repeat rate
bar_colors = ['#2ecc71' if v > global_rate / 100 else '#e74c3c'
              for v in top_n['repeat_rate']]
bars = ax1.barh(top_n['first_category'], top_n['repeat_rate_pct'],
                color=bar_colors, edgecolor='none', height=0.65)
ax1.axvline(global_rate, color='black', linestyle='--', linewidth=1,
            label=f'Overall: {global_rate:.1f}%')
for bar, row in zip(bars, top_n.itertuples()):
    ax1.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
             f'n={row.customer_count:,}', va='center', fontsize=7.5)
ax1.set_xlabel('Repeat Purchase Rate (%)', fontsize=11)
ax1.set_title('Top 20 Categories by Repeat Rate', fontsize=11, fontweight='bold')
ax1.legend(frameon=False, fontsize=9)
ax1.grid(True, axis='x', linestyle='--', alpha=0.4)
ax1.spines[['top', 'right']].set_visible(False)

# Bottom 10 — lowest repeat rate (churn risk categories)
bars2 = ax2.barh(bot_n['first_category'], bot_n['repeat_rate_pct'],
                 color='#e74c3c', edgecolor='none', height=0.65, alpha=0.8)
ax2.axvline(global_rate, color='black', linestyle='--', linewidth=1,
            label=f'Overall: {global_rate:.1f}%')
for bar, row in zip(bars2, bot_n.itertuples()):
    ax2.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
             f'n={row.customer_count:,}', va='center', fontsize=7.5)
ax2.set_xlabel('Repeat Purchase Rate (%)', fontsize=11)
ax2.set_title('Bottom 10 Categories by Repeat Rate', fontsize=11, fontweight='bold')
ax2.legend(frameon=False, fontsize=9)
ax2.grid(True, axis='x', linestyle='--', alpha=0.4)
ax2.spines[['top', 'right']].set_visible(False)

plt.suptitle(
    'Repeat Purchase Rate by First Purchase Category  (min 50 customers)',
    fontsize=13, fontweight='bold',
)
plt.tight_layout()
plt.savefig(FIG_DIR / '03_category_repeat_rate.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Repeat rate vs avg total spend — shows which categories produce high-value loyalists
plot_df = cat_retention[cat_retention['customer_count'] >= 100].copy()
size_norm = (plot_df['customer_count'] - plot_df['customer_count'].min()) / \
            (plot_df['customer_count'].max() - plot_df['customer_count'].min())

fig, ax = plt.subplots(figsize=(11, 6))
sc = ax.scatter(
    plot_df['repeat_rate_pct'],
    plot_df['avg_total_spend'],
    s=40 + size_norm * 300,
    c=plot_df['avg_orders'],
    cmap='YlGn', alpha=0.8, edgecolors='grey', linewidths=0.4,
)
plt.colorbar(sc, ax=ax, label='Avg Orders per Customer')
ax.axvline(global_rate, color='grey', linestyle='--', linewidth=1, alpha=0.6)

for _, row in plot_df.nlargest(8, 'avg_total_spend').iterrows():
    ax.annotate(row['first_category'],
                (row['repeat_rate_pct'], row['avg_total_spend']),
                fontsize=7, xytext=(4, 3), textcoords='offset points')

ax.set_xlabel('Repeat Purchase Rate (%)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'R${v:,.0f}'))
ax.set_ylabel('Avg Total Customer Spend', fontsize=11)
ax.set_title('Category Loyalty vs Revenue Value  (size = customer count)',
             fontsize=12, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / '03_category_loyalty_vs_spend.png', dpi=150, bbox_inches='tight')
plt.show()